In [1]:
from google.colab import files

uploaded = files.upload()

Saving portuguese_test_processed.csv to portuguese_test_processed.csv
Saving portuguese_train_processed.csv to portuguese_train_processed.csv


Step 2 – Import required libraries

In [2]:
import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

from sklearn.model_selection import (
    KFold,
    cross_validate,
    cross_val_predict
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


Step 3 – Create output folders in Google Colab

In [3]:
project_dir = Path("/content/Portuguese_ML_Project")

models_dir = project_dir / "models"

results_dir = (
    project_dir / "output" /
    "linear_regression" / "comparison"
)

models_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print("Output directory:", results_dir)

Output directory: /content/Portuguese_ML_Project/output/linear_regression/comparison


Step 4 – Load the processed datasets

In [5]:
train_df = pd.read_csv(
    "/content/portuguese_train_processed.csv"
)

test_df = pd.read_csv(
    "/content/portuguese_test_processed.csv"
)

X_train = train_df.drop(columns=["G3"])
y_train = train_df["G3"]

X_test = test_df.drop(columns=["G3"])
y_test = test_df["G3"]

assert X_train.columns.equals(X_test.columns)
assert X_train.notna().all().all()
assert X_test.notna().all().all()

print("Training students:", len(X_train))
print("Testing students:", len(X_test))
print("Input features:", X_train.shape[1])

display(train_df.head())

Training students: 519
Testing students: 130
Input features: 58


,num__age,num__Medu,num__Fedu,num__traveltime,num__studytime,num__failures,num__famrel,num__freetime,num__goout,num__Dalc,...,cat__activities_yes,cat__nursery_no,cat__nursery_yes,cat__higher_no,cat__higher_yes,cat__internet_no,cat__internet_yes,cat__romantic_no,cat__romantic_yes,G3
0,0.987932,-0.437418,-0.256337,-0.754310,1.352962,-0.382133,0.068136,-0.186680,-0.196270,-0.539674,...,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,18
1,-0.629534,1.350140,1.591423,-0.754310,0.121054,-0.382133,0.068136,0.753970,1.501468,3.579309,...,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,12
2,0.987932,0.456361,-0.256337,-0.754310,1.352962,-0.382133,1.108214,-0.186680,-1.045139,-0.539674,...,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,12
3,0.179199,-0.437418,-1.180216,-0.754310,-1.110853,-0.382133,0.068136,0.753970,-1.045139,0.490072,...,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,13
4,0.179199,-1.331196,-1.180216,0.555011,-1.110853,-0.382133,-0.971942,1.694619,1.501468,0.490072,...,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,10


Step 5 – Define three Linear Regression experiments

In [6]:
grade_cols = ["num__G1", "num__G2"]

assert all(col in X_train.columns for col in grade_cols)

other_features = [
    col for col in X_train.columns
    if col not in grade_cols
]

experiments = {
    "LR-1 All Features": X_train.columns.tolist(),
    "LR-2 G1 and G2": grade_cols,
    "LR-3 Without G1/G2": other_features
}

experiment_table = pd.DataFrame([
    {
        "Experiment": name,
        "Number of Features": len(features)
    }
    for name, features in experiments.items()
])

display(experiment_table)

,Experiment,Number of Features
0,LR-1 All Features,58
1,LR-2 G1 and G2,2
2,LR-3 Without G1/G2,56


Step 6 – Configure cross-validation

In [7]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

print("5-Fold Cross-Validation configured!")

5-Fold Cross-Validation configured!


Step 7 – Train all three Linear Regression configurations

In [8]:
lr_results = []
lr_fold_results = []
prediction_results = {}

actual = y_train.to_numpy()
actual_binary = (actual >= 10).astype(int)

for name, features in experiments.items():

    print("Training:", name)

    X_selected = X_train[features]

    model = LinearRegression()

    scores = cross_validate(
        model,
        X_selected,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        error_score="raise"
    )

    predictions = cross_val_predict(
        LinearRegression(),
        X_selected,
        y_train,
        cv=cv
    )

    prediction_results[name] = predictions

    rounded_predictions = np.clip(
        np.rint(predictions), 0, 20
    ).astype(int)

    predicted_binary = (
        predictions >= 10
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        actual_binary,
        predicted_binary,
        labels=[0, 1]
    ).ravel()

    train_rmse = np.sqrt(
        -scores["train_MSE"]
    ).mean()

    val_rmse = np.sqrt(
        -scores["test_MSE"]
    ).mean()

    lr_results.append({
        "Experiment": name,
        "Features": len(features),

        "Train MAE": -scores["train_MAE"].mean(),
        "Validation MAE": -scores["test_MAE"].mean(),

        "Train MSE": -scores["train_MSE"].mean(),
        "Validation MSE": -scores["test_MSE"].mean(),

        "Train RMSE": train_rmse,
        "Validation RMSE": val_rmse,

        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),

        "RMSE Gap": val_rmse - train_rmse,

        "Exact Grade Accuracy (%)": (
            np.mean(rounded_predictions == actual) * 100
        ),

        "Within 1 Grade (%)": (
            np.mean(np.abs(actual - predictions) <= 1) * 100
        ),

        "Within 2 Grades (%)": (
            np.mean(np.abs(actual - predictions) <= 2) * 100
        ),

        "Pass/Fail Accuracy (%)": (
            accuracy_score(
                actual_binary,
                predicted_binary
            ) * 100
        ),

        "Precision": precision_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Recall": recall_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "Specificity": (
            tn / (tn + fp)
            if (tn + fp) > 0 else 0
        ),

        "F1 Score": f1_score(
            actual_binary,
            predicted_binary,
            zero_division=0
        ),

        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

    for fold in range(5):
        lr_fold_results.append({
            "Experiment": name,
            "Fold": fold + 1,

            "Train MAE": -scores["train_MAE"][fold],
            "Validation MAE": -scores["test_MAE"][fold],

            "Train RMSE": np.sqrt(
                -scores["train_MSE"][fold]
            ),

            "Validation RMSE": np.sqrt(
                -scores["test_MSE"][fold]
            ),

            "Train R2": scores["train_R2"][fold],
            "Validation R2": scores["test_R2"][fold]
        })

lr_results_df = pd.DataFrame(lr_results)

lr_results_df = lr_results_df.sort_values(
    ["Validation RMSE", "Validation MAE", "RMSE Gap"],
    ascending=[True, True, True]
).reset_index(drop=True)

lr_fold_df = pd.DataFrame(lr_fold_results)

print("All three Linear Regression experiments completed!")

Training: LR-1 All Features
Training: LR-2 G1 and G2
Training: LR-3 Without G1/G2
All three Linear Regression experiments completed!


Step 8 – Regression error comparison

In [9]:
regression_columns = [
    "Experiment",
    "Features",
    "Train MAE",
    "Validation MAE",
    "Train MSE",
    "Validation MSE",
    "Train RMSE",
    "Validation RMSE",
    "Train R2",
    "Validation R2",
    "RMSE Gap"
]

print("=" * 65)
print("PORTUGUESE – LINEAR REGRESSION ERROR COMPARISON")
print("=" * 65)

display(
    lr_results_df[regression_columns].round(4)
)

PORTUGUESE – LINEAR REGRESSION ERROR COMPARISON


,Experiment,Features,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap
0,LR-2 G1 and G2,2,0.8058,0.8163,1.6440,1.6889,1.2798,1.2720,0.8441,0.8417,-0.0078
1,LR-1 All Features,58,0.7914,0.8846,1.4544,1.8437,1.2036,1.3322,0.8621,0.8261,0.1286
2,LR-3 Without G1/G2,56,1.8248,2.0456,6.2969,7.8747,2.5083,2.7897,0.4008,0.2256,0.2815


Step 9 – Grade accuracy comparison

In [10]:
grade_columns = [
    "Experiment",
    "Exact Grade Accuracy (%)",
    "Within 1 Grade (%)",
    "Within 2 Grades (%)"
]

print("=" * 65)
print("PORTUGUESE – GRADE ACCURACY COMPARISON")
print("=" * 65)

display(
    lr_results_df[grade_columns].round(2)
)

PORTUGUESE – GRADE ACCURACY COMPARISON


,Experiment,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%)
0,LR-2 G1 and G2,42.20,77.84,95.57
1,LR-1 All Features,39.11,70.71,93.83
2,LR-3 Without G1/G2,18.11,33.72,58.77


Step 10 – Pass/fail classification comparison

In [11]:
classification_columns = [
    "Experiment",
    "Pass/Fail Accuracy (%)",
    "Precision",
    "Recall",
    "Specificity",
    "F1 Score"
]

print("=" * 65)
print("PORTUGUESE – CLASSIFICATION COMPARISON")
print("=" * 65)

display(
    lr_results_df[classification_columns].round(4)
)

PORTUGUESE – CLASSIFICATION COMPARISON


,Experiment,Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score
0,LR-2 G1 and G2,87.8613,0.9818,0.8710,0.9176,0.9231
1,LR-1 All Features,87.6686,0.9818,0.8687,0.9176,0.9218
2,LR-3 Without G1/G2,81.6956,0.8933,0.8871,0.4588,0.8902


Step 11 – Confusion matrix comparison table

In [12]:
confusion_columns = [
    "Experiment",
    "TN",
    "FP",
    "FN",
    "TP"
]

print("=" * 65)
print("PORTUGUESE – CONFUSION MATRIX COMPARISON")
print("=" * 65)

display(
    lr_results_df[confusion_columns]
)

PORTUGUESE – CONFUSION MATRIX COMPARISON


,Experiment,TN,FP,FN,TP
0,LR-2 G1 and G2,78,7,56,378
1,LR-1 All Features,78,7,57,377
2,LR-3 Without G1/G2,39,46,49,385


Step 12 – Overfitting and underfitting assessment

In [13]:
baseline_scores = cross_validate(
    DummyRegressor(strategy="mean"),
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

baseline_rmse = np.sqrt(
    -baseline_scores["test_MSE"]
).mean()

def assess_fitting(row):

    gap_ratio = (
        row["RMSE Gap"] / row["Train RMSE"]
        if row["Train RMSE"] > 0
        else np.inf
    )

    if row["Validation RMSE"] >= baseline_rmse:
        return "Weak performance / possible underfitting"

    elif gap_ratio > 0.30:
        return "Possible overfitting"

    else:
        return "Reasonable generalization"

lr_results_df["Fitting Assessment"] = (
    lr_results_df.apply(assess_fitting, axis=1)
)

print("Baseline RMSE:", round(baseline_rmse, 4))

display(
    lr_results_df[
        [
            "Experiment",
            "Train RMSE",
            "Validation RMSE",
            "RMSE Gap",
            "Fitting Assessment"
        ]
    ].round(4)
)

Baseline RMSE: 3.2326


,Experiment,Train RMSE,Validation RMSE,RMSE Gap,Fitting Assessment
0,LR-2 G1 and G2,1.2798,1.2720,-0.0078,Reasonable generalization
1,LR-1 All Features,1.2036,1.3322,0.1286,Reasonable generalization
2,LR-3 Without G1/G2,2.5083,2.7897,0.2815,Reasonable generalization


Step 13 – Final complete comparison table

In [14]:
final_comparison = lr_results_df.copy()

final_comparison.insert(
    0,
    "Rank",
    np.arange(1, len(final_comparison) + 1)
)

print("=" * 70)
print("FINAL PORTUGUESE LINEAR REGRESSION COMPARISON")
print("=" * 70)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 250
):
    display(final_comparison.round(4))

FINAL PORTUGUESE LINEAR REGRESSION COMPARISON


,Rank,Experiment,Features,Train MAE,Validation MAE,Train MSE,Validation MSE,Train RMSE,Validation RMSE,Train R2,Validation R2,RMSE Gap,Exact Grade Accuracy (%),Within 1 Grade (%),Within 2 Grades (%),Pass/Fail Accuracy (%),Precision,Recall,Specificity,F1 Score,TN,FP,FN,TP,Fitting Assessment
0,1,LR-2 G1 and G2,2,0.8058,0.8163,1.6440,1.6889,1.2798,1.2720,0.8441,0.8417,-0.0078,42.1965,77.8420,95.5684,87.8613,0.9818,0.8710,0.9176,0.9231,78,7,56,378,Reasonable generalization
1,2,LR-1 All Features,58,0.7914,0.8846,1.4544,1.8437,1.2036,1.3322,0.8621,0.8261,0.1286,39.1137,70.7129,93.8343,87.6686,0.9818,0.8687,0.9176,0.9218,78,7,57,377,Reasonable generalization
2,3,LR-3 Without G1/G2,56,1.8248,2.0456,6.2969,7.8747,2.5083,2.7897,0.4008,0.2256,0.2815,18.1118,33.7187,58.7669,81.6956,0.8933,0.8871,0.4588,0.8902,39,46,49,385,Reasonable generalization


Step 14 – Select the best Linear Regression configuration

In [15]:
best_result = final_comparison.iloc[0]

best_name = best_result["Experiment"]
best_features = experiments[best_name]

print("=" * 60)
print("BEST PORTUGUESE LINEAR REGRESSION CONFIGURATION")
print("=" * 60)

print("Experiment:", best_name)
print("Number of features:", len(best_features))

print("\nREGRESSION METRICS")
print(f"Validation RMSE : {best_result['Validation RMSE']:.4f}")
print(f"Validation MAE  : {best_result['Validation MAE']:.4f}")
print(f"Validation R²   : {best_result['Validation R2']:.4f}")
print(f"RMSE Gap        : {best_result['RMSE Gap']:.4f}")

print("\nGRADE ACCURACY")
print(f"Exact Accuracy  : {best_result['Exact Grade Accuracy (%)']:.2f}%")
print(f"Within ±1 Grade : {best_result['Within 1 Grade (%)']:.2f}%")
print(f"Within ±2 Grades: {best_result['Within 2 Grades (%)']:.2f}%")

print("\nPASS/FAIL CLASSIFICATION")
print(f"Accuracy   : {best_result['Pass/Fail Accuracy (%)']:.2f}%")
print(f"Precision  : {best_result['Precision']:.4f}")
print(f"Recall     : {best_result['Recall']:.4f}")
print(f"Specificity: {best_result['Specificity']:.4f}")
print(f"F1 Score   : {best_result['F1 Score']:.4f}")

print("\nCONFUSION MATRIX")
print("TN:", int(best_result["TN"]))
print("FP:", int(best_result["FP"]))
print("FN:", int(best_result["FN"]))
print("TP:", int(best_result["TP"]))

print("\nFITTING ASSESSMENT")
print(best_result["Fitting Assessment"])

BEST PORTUGUESE LINEAR REGRESSION CONFIGURATION
Experiment: LR-2 G1 and G2
Number of features: 2

REGRESSION METRICS
Validation RMSE : 1.2720
Validation MAE  : 0.8163
Validation R²   : 0.8417
RMSE Gap        : -0.0078

GRADE ACCURACY
Exact Accuracy  : 42.20%
Within ±1 Grade : 77.84%
Within ±2 Grades: 95.57%

PASS/FAIL CLASSIFICATION
Accuracy   : 87.86%
Precision  : 0.9818
Recall     : 0.8710
Specificity: 0.9176
F1 Score   : 0.9231

CONFUSION MATRIX
TN: 78
FP: 7
FN: 56
TP: 378

FITTING ASSESSMENT
Reasonable generalization
